# Training Frame Exploration

Fresh EDA on `build_training_frame`'s output -- the actual dataframe every moneyline
model comparison in `model_backtest_results.ipynb` has been run against this session
(EPA/Elo/QB features, `market_prob_home`, `home_win`). Two goals: (1) see how it's
actually structured (dtypes, nulls, ranges), (2) look at the CORRELATION STRUCTURE
among existing features as a starting point for feature engineering ideas.

**Not to be confused with `correlated_props.py`'s "correlated" work** -- that's
same-game OUTCOME correlation (does team_won co-occur with a rushing/passing
outcome). This notebook is about FEATURE correlation / multicollinearity among the
model's own inputs -- a different, more standard EDA question, kept in its own
notebook so the two "correlated" senses don't get conflated.

Deliberately exploratory in tone, matching `model_backtest_results.ipynb`'s own
established style -- surfacing structure for the next real feature-engineering
attempt, not claiming a finding.


In [1]:
from pathlib import Path

ROOT = Path("..").resolve()
assert (ROOT / "pyproject.toml").exists(), f"unexpected repo root: {ROOT}"

import numpy as np
import pandas as pd

from src.model.elo import compute_team_elo_ratings
from src.model.qb_elo import compute_qb_elo_ratings
from src.model.train import (
    COMBINED_QB_FEATURE_COLUMNS,
    ELO_FEATURE_COLUMNS,
    FEATURE_COLUMNS,
    QB_FEATURE_COLUMNS,
    build_training_frame,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


## Load real cached data and build the training frame

Same real data (`schedules_2015_2025.parquet`, `pbp_2015_2025.parquet`) and the same
`build_training_frame` function every model comparison this session has used -- not
a new data source.


In [2]:
schedule = pd.read_parquet(ROOT / "data" / "static" / "schedules_2015_2025.parquet")
pbp = pd.read_parquet(ROOT / "data" / "static" / "pbp_2015_2025.parquet")

elo_ratings = compute_team_elo_ratings(schedule)
qb_elo_ratings = compute_qb_elo_ratings(schedule, elo_ratings)

df = build_training_frame(pbp, schedule, elo_ratings=elo_ratings, qb_elo_ratings=qb_elo_ratings)
print(f"{len(df)} games with a usable label + full feature set (out of {len(schedule)} total games)")
df.head()


2879 games with a usable label + full feature set (out of 3028 total games)


,game_id,season,week,gameday,weekday,home_team,away_team,home_rest,away_rest,div_game,roof,surface,temp,wind,spread_line,total_line,stadium_id,location,home_pre_off_epa,home_pre_off_success,home_pre_def_epa_allowed,home_pre_def_success_allowed,away_pre_off_epa,away_pre_off_success,away_pre_def_epa_allowed,away_pre_def_success_allowed,home_score,away_score,home_moneyline,away_moneyline,home_pre_elo,away_pre_elo,home_pre_qb_elo,away_pre_qb_elo,home_win,market_prob_home
16,2015_02_DEN_KC,2015,2,2015-09-17,Thursday,KC,DEN,4,4,1,outdoors,grass,87.0,13.0,3.0,42.0,KAN00,Home,0.002278,0.432836,-0.130424,0.453333,-0.227019,0.333333,-0.362337,0.293103,24,31,-162.0,146.0,1524.176621,1516.429246,1509.670648,1506.571698,0,0.603343
17,2015_02_NE_BUF,2015,2,2015-09-20,Sunday,BUF,NE,7,10,1,outdoors,a_turf,62.0,7.0,2.0,45.0,BUF00,Home,0.080444,0.357143,0.009576,0.405405,0.290281,0.533333,0.249406,0.513889,32,40,-124.0,112.0,1522.281461,1517.556646,1508.912584,1507.022658,0,0.539928
18,2015_02_HOU_CAR,2015,2,2015-09-20,Sunday,CAR,HOU,7,7,0,outdoors,grass,88.0,6.0,3.0,41.0,CAR00,Home,-0.088671,0.362319,-0.318733,0.378788,-0.130424,0.453333,0.002278,0.432836,24,17,-143.0,129.0,1528.890759,1475.823379,1511.556304,1500.000000,1,0.574035
19,2015_02_ARI_CHI,2015,2,2015-09-20,Sunday,CHI,ARI,7,7,0,outdoors,grass,70.0,6.0,-2.0,45.5,CHI98,Home,0.093701,0.458333,0.391846,0.561404,0.210083,0.459016,-0.020548,0.369863,23,48,108.0,-119.0,1474.453975,1521.655770,1489.781590,1508.662308,0,0.469433
21,2015_02_TEN_CLE,2015,2,2015-09-20,Sunday,CLE,TEN,7,7,0,outdoors,grass,64.0,12.0,1.0,41.0,CLE00,Home,-0.237009,0.388060,0.148887,0.457627,0.301102,0.538462,-0.285730,0.328571,28,14,-113.0,102.0,1473.902446,1539.149854,1500.000000,1515.659942,1,0.517291


## Structure: dtypes, null rates, ranges

The literal "how is it structured" question -- shape, column types, and how many
rows would be lost per column if used as-is (leakage-guard rows near each team's
first game in the window show up here as real, expected nulls, not a data bug --
see `build_training_frame`'s own docstring).


In [3]:
print(f"shape: {df.shape}")
print(f"\nseasons covered: {sorted(df['season'].unique())}")
print(f"\ndtypes:\n{df.dtypes}")


shape: (2879, 36)

seasons covered: [np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]

dtypes:
game_id                             str
season                            int32
week                              int32
gameday                             str
weekday                             str
home_team                           str
away_team                           str
home_rest                         int32
away_rest                         int32
div_game                          int32
roof                                str
surface                             str
temp                            float64
wind                            float64
spread_line                     float64
total_line                      float64
stadium_id                          str
location                            str
home_pre_off_epa                float64
home_pre_off_succe

In [4]:
null_report = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_null": df.isna().sum(),
    "pct_null": (df.isna().mean() * 100).round(2),
})
null_report[null_report["n_null"] > 0].sort_values("n_null", ascending=False)


,dtype,n_null,pct_null
temp,float64,1005,34.91
wind,float64,1005,34.91


In [5]:
feature_cols = COMBINED_QB_FEATURE_COLUMNS  # EPA + Elo + QB Elo -- this project's current practical feature set
df[feature_cols + ["market_prob_home", "home_win"]].describe().T


,count,mean,std,min,25%,50%,75%,max
home_pre_off_epa,2879.0,-0.001311,0.112066,-0.728351,-0.069716,0.005306,0.073093,0.394506
home_pre_off_success,2879.0,0.436432,0.040913,0.152174,0.410803,0.437058,0.464307,0.615385
home_pre_def_epa_allowed,2879.0,0.010680,0.097909,-0.497868,-0.048204,0.012742,0.066260,0.466531
home_pre_def_success_allowed,2879.0,0.439660,0.036575,0.264706,0.416101,0.438641,0.462775,0.619048
away_pre_off_epa,2879.0,-0.003826,0.107234,-0.559477,-0.072744,0.001336,0.066842,0.395585
away_pre_off_success,2879.0,0.435664,0.040446,0.189655,0.410541,0.436578,0.462672,0.580128
away_pre_def_epa_allowed,2879.0,0.010098,0.101441,-0.728351,-0.047668,0.011084,0.065486,0.722965
away_pre_def_success_allowed,2879.0,0.439068,0.037843,0.189655,0.415994,0.438423,0.460979,0.679245
home_rest,2879.0,7.485585,2.128854,4.000000,7.000000,7.000000,7.000000,16.000000
away_rest,2879.0,7.470302,2.061697,4.000000,7.000000,7.000000,7.000000,17.000000


## Correlation structure among existing features

The actual "start some correlated feature engineering" ask: how correlated are the
model's own inputs with each other and with the label? High correlation between two
features (e.g. base Elo and QB Elo) would mean one is redundant given the other, not
adding independent signal; low correlation between a feature and `home_win` is a
weak candidate on its own, though it could still help in combination.


In [6]:
corr_cols = feature_cols + ["market_prob_home", "home_win"]
corr = df[corr_cols].corr()
corr.style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1).format("{:.2f}")


,home_pre_off_epa,home_pre_off_success,home_pre_def_epa_allowed,home_pre_def_success_allowed,away_pre_off_epa,away_pre_off_success,away_pre_def_epa_allowed,away_pre_def_success_allowed,home_rest,away_rest,div_game,home_pre_elo,away_pre_elo,home_pre_qb_elo,away_pre_qb_elo,market_prob_home,home_win
home_pre_off_epa,1.00,0.83,0.05,0.23,0.07,0.06,0.02,0.05,0.04,0.04,-0.03,0.58,0.04,0.41,0.04,0.40,0.15
home_pre_off_success,0.83,1.00,0.14,0.28,0.08,0.08,0.03,0.06,0.05,0.05,-0.02,0.50,0.03,0.37,0.04,0.35,0.14
home_pre_def_epa_allowed,0.05,0.14,1.00,0.76,0.01,0.02,0.06,0.06,-0.01,0.00,-0.00,-0.30,-0.03,-0.13,-0.00,-0.20,-0.07
home_pre_def_success_allowed,0.23,0.28,0.76,1.00,0.02,0.03,0.07,0.10,-0.01,0.01,-0.01,-0.13,-0.04,-0.03,0.00,-0.09,-0.02
away_pre_off_epa,0.07,0.08,0.01,0.02,1.00,0.82,0.09,0.23,0.01,0.04,-0.01,0.06,0.58,0.05,0.39,-0.38,-0.14
away_pre_off_success,0.06,0.08,0.02,0.03,0.82,1.00,0.16,0.27,0.00,0.02,0.01,0.05,0.49,0.05,0.34,-0.34,-0.15
away_pre_def_epa_allowed,0.02,0.03,0.06,0.07,0.09,0.16,1.00,0.79,-0.03,-0.01,0.02,-0.03,-0.29,-0.00,-0.11,0.16,0.01
away_pre_def_success_allowed,0.05,0.06,0.06,0.10,0.23,0.27,0.79,1.00,-0.03,0.01,0.01,-0.01,-0.14,0.03,-0.04,0.07,-0.02
home_rest,0.04,0.05,-0.01,-0.01,0.01,0.00,-0.03,-0.03,1.00,0.29,-0.06,0.06,0.04,0.05,0.02,0.06,0.02
away_rest,0.04,0.05,0.00,0.01,0.04,0.02,-0.01,0.01,0.29,1.00,-0.08,0.02,0.03,-0.00,0.01,-0.04,-0.03


In [7]:
# Feature-vs-label correlation, ranked -- which existing features are individually
# most associated with the outcome (not the same as "most useful in a model", which
# depends on what else is already included, but a real starting point).
label_corr = corr["home_win"].drop("home_win").sort_values(key=abs, ascending=False)
label_corr


market_prob_home                0.383592
home_pre_elo                    0.225943
away_pre_elo                   -0.206209
home_pre_qb_elo                 0.175423
home_pre_off_epa                0.151744
away_pre_qb_elo                -0.150452
away_pre_off_success           -0.147267
away_pre_off_epa               -0.142582
home_pre_off_success            0.138226
home_pre_def_epa_allowed       -0.073003
away_rest                      -0.032404
home_pre_def_success_allowed   -0.024230
away_pre_def_success_allowed   -0.018586
div_game                       -0.017645
home_rest                       0.016786
away_pre_def_epa_allowed        0.010303
Name: home_win, dtype: float64

In [8]:
# home_pre_elo vs home_pre_qb_elo -- are these two ratings capturing genuinely
# different signal, or is QB Elo mostly redundant with team Elo? (qb_elo.py's own
# design is a RESIDUAL rating specifically meant to isolate QB skill from team
# strength -- this checks whether that design goal actually shows up as low
# correlation in practice, on the real full 2015-2025 cache.)
elo_qb_corr = df[["home_pre_elo", "home_pre_qb_elo", "away_pre_elo", "away_pre_qb_elo"]].corr()
elo_qb_corr


,home_pre_elo,home_pre_qb_elo,away_pre_elo,away_pre_qb_elo
home_pre_elo,1.000000,0.648922,0.084281,0.067483
home_pre_qb_elo,0.648922,1.000000,0.074374,0.094315
away_pre_elo,0.084281,0.074374,1.000000,0.634278
away_pre_qb_elo,0.067483,0.094315,0.634278,1.000000


## Candidate interaction terms

A quick look at whether combining two existing features (multiplicatively, as an
interaction term) correlates with the outcome more strongly than either alone --
genuinely different information from a pairwise linear correlation, a real
candidate feature-engineering direction. Exploratory only -- not a claim, a
starting point for an actual comparison in `model_backtest_results.ipynb` if one of
these looks promising.


In [9]:
elo_diff = df["home_pre_elo"] - df["away_pre_elo"]
qb_elo_diff = df["home_pre_qb_elo"] - df["away_pre_qb_elo"]
epa_diff = df["home_pre_off_epa"] - df["away_pre_off_epa"]

candidates = pd.DataFrame({
    "elo_diff": elo_diff,
    "qb_elo_diff": qb_elo_diff,
    "epa_diff": epa_diff,
    "elo_diff_x_qb_elo_diff": elo_diff * qb_elo_diff,
    "rest_diff": df["home_rest"] - df["away_rest"],
    "home_win": df["home_win"].astype(int),
})
candidates.corr()["home_win"].drop("home_win").sort_values(key=abs, ascending=False)


elo_diff                  0.319527
qb_elo_diff               0.242311
epa_diff                  0.215420
rest_diff                 0.040947
elo_diff_x_qb_elo_diff    0.014771
Name: home_win, dtype: float64